<a href="https://colab.research.google.com/github/lauandsalih/Monte-Carlo-simulation/blob/main/inventory_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_inventory(reorder_point, order_quantity, days=365, lambda_demand=5, lead_time=3, holding_cost=1, stockout_cost=10, order_cost=50):
    """
    Simulates warehouse inventory over a set period with stochastic demand.
    Returns the total cost incurred over the period.
    """
    inventory = reorder_point + order_quantity
    total_cost = 0

    days_until_delivery = 0
    order_in_transit = False

    for day in range(days):
        # 1. Process arriving deliveries
        if order_in_transit:
            days_until_delivery -= 1
            if days_until_delivery <= 0:
                inventory += order_quantity
                order_in_transit = False

        # 2. Simulate daily demand (Stochastic Poisson Process)
        demand = np.random.poisson(lambda_demand)

        # 3. Fulfill demand and calculate stockouts
        if inventory >= demand:
            inventory -= demand
        else:
            stockout_amount = demand - inventory
            total_cost += stockout_amount * stockout_cost
            inventory = 0  # Assuming lost sales, not backordered

        # 4. Calculate daily holding costs
        total_cost += inventory * holding_cost

        # 5. Trigger new order if below threshold
        if inventory <= reorder_point and not order_in_transit:
            order_in_transit = True
            days_until_delivery = lead_time
            total_cost += order_cost

    return total_cost

def optimize_reorder_point(simulations=100, max_reorder_test=30):
    """
    Runs multiple Monte Carlo simulations across different reorder points
    to find the optimal threshold that minimizes expected total cost.
    """
    results = []
    reorder_points_to_test = range(0, max_reorder_test)

    # Fixed parameters for this scenario
    fixed_q = 20

    print("Running Monte Carlo simulations...")
    for s in reorder_points_to_test:
        costs_for_s = []
        # Run multiple paths to account for stochastic noise
        for _ in range(simulations):
            cost = simulate_inventory(reorder_point=s, order_quantity=fixed_q)
            costs_for_s.append(cost)

        expected_cost = np.mean(costs_for_s)
        results.append((s, expected_cost))

    # Convert to DataFrame for easy analysis
    df = pd.DataFrame(results, columns=['Reorder_Point', 'Expected_Total_Cost'])

    # Find the optimal point
    optimal_s = df.loc[df['Expected_Total_Cost'].idxmin()]

    print(f"Optimization Complete.")
    print(f"Optimal Reorder Point (s): {optimal_s['Reorder_Point']}")
    print(f"Minimum Expected Annual Cost: €{optimal_s['Expected_Total_Cost']:.2f}")

    return df

if __name__ == "__main__":
    # Ensure reproducibility
    np.random.seed(42)

    # Run optimization
    optimization_results = optimize_reorder_point(simulations=200, max_reorder_test=40)

Running Monte Carlo simulations...
Optimization Complete.
Optimal Reorder Point (s): 13.0
Minimum Expected Annual Cost: €8773.22
